In [7]:
import cv2

# Camera indices - usually 0 and 1, but may differ if you have
# a built-in webcam or other capture devices. Adjust if needed.
CAM1_INDEX = 0
CAM2_INDEX = 1

def open_camera(index):
    # CAP_DSHOW (DirectShow) is important on Windows -
    # the default MSMF backend often has problems with
    # multiple USB cameras at once
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print(f"ERROR: Could not open camera {index}")
        return None
    # Optional: reduce resolution to lower USB bandwidth
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    print(f"Camera {index} opened: "
          f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x"
          f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
    return cap

cap1 = open_camera(CAM1_INDEX)
cap2 = open_camera(CAM2_INDEX)

if cap1 and cap2:
    print("Both cameras opened. Press 'q' to quit.")
    while True:
        ok1, frame1 = cap1.read()
        ok2, frame2 = cap2.read()

        if not ok1:
            print("Camera 1: frame grab failed")
        if not ok2:
            print("Camera 2: frame grab failed")
        if not ok1 or not ok2:
            break

        cv2.imshow("Microscope 1", frame1)
        cv2.imshow("Microscope 2", frame2)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

if cap1: cap1.release()
if cap2: cap2.release()
cv2.destroyAllWindows()

Camera 0 opened: 640x480
Camera 1 opened: 640x480
Both cameras opened. Press 'q' to quit.
Camera 1: frame grab failed


In [5]:
import cv2
for i in range(5):
    cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
    if cap.isOpened():
        print(f"Index {i}: available")
        cap.release()

Index 0: available
Index 1: available


In [1]:
import cv2

class OnDemandCamera:
    def __init__(self, index):
        self.index = index
        self.cap = None

    def activate(self):
        if self.cap is None:
            self.cap = cv2.VideoCapture(self.index, cv2.CAP_DSHOW)
            if not self.cap.isOpened():
                self.cap = None
                raise RuntimeError(f"Camera {self.index} could not be opened")
            # discard the first few frames - many cameras need
            # a moment for auto-exposure to settle
            for _ in range(5):
                self.cap.read()
        return self.cap

    def snapshot(self):
        cap = self.activate()
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"Camera {self.index}: frame grab failed")
        return frame

    def deactivate(self):
        if self.cap is not None:
            self.cap.release()
            self.cap = None


cam1 = OnDemandCamera(0)
cam2 = OnDemandCamera(1)



In [3]:

# ... camera is idle here ...



frame = cam1.snapshot()      # activates, grabs image
cv2.imwrite("scope1.png", frame)
cam1.deactivate()            # back to standby

# ... idle again ...